In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
!pip install -q transformers datasets accelerate bitsandbytes torchcodec librosa peft evaluate

# Import & Data Loading

In [ ]:
import os, torch, gc, zipfile, glob
import pandas as pd
from datasets import Dataset, Audio
from huggingface_hub import hf_hub_download, login
from transformers import (
    Wav2Vec2Processor, 
    Wav2Vec2ForCTC, 
    BitsAndBytesConfig, 
    TrainingArguments, 
    Trainer, 
    TrainerCallback
)
from peft import (
    LoraConfig, 
    get_peft_model, 
    prepare_model_for_kbit_training,
    PeftModel
)

BASE_DIR = "/kaggle/working/khotbah_dataset"
os.makedirs(BASE_DIR, exist_ok=True)

login("")

REPO_ID = "gracecalista/new-dataset-transcribe"
TOTAL_PARTS = 2

# Download Metadata
print("Mendownload metadata...")
csv_path = hf_hub_download(
    repo_id=REPO_ID,
    filename="benchmarking_results_final.csv",
    repo_type="dataset",
    local_dir=BASE_DIR
)
df = pd.read_csv(csv_path).dropna(subset=['text', 'path'])

# Download & Extract semua part
for part_num in range(1, TOTAL_PARTS + 1):
    EXTRACT_FLAG = os.path.join(BASE_DIR, f".extracted_part{part_num}")
    zip_filename = f"wavs/wavs_part{part_num}.zip"

    if os.path.exists(EXTRACT_FLAG):
        with open(EXTRACT_FLAG, 'r') as f:
            flag_content = f.read()
        print(f"Part {part_num}: Sudah diekstrak sebelumnya ({flag_content}), skip.")
        continue

    print(f"\n [{part_num}/{TOTAL_PARTS}] Mendownload {zip_filename}...")
    try:
        zip_path = hf_hub_download(
            repo_id=REPO_ID,
            filename=zip_filename,
            repo_type="dataset",
            local_dir=BASE_DIR
        )
    except Exception as e:
        print(f" Part {part_num} gagal didownload: {e}")
        continue

    print(f"Download selesai. Mengekstrak dari: {zip_path}")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        members = zip_ref.infolist()
        total = len(members)
        print(f"Total file dalam zip: {total}")
        for i, member in enumerate(members, 1):
            zip_ref.extract(member, BASE_DIR)
            if i % 5000 == 0 or i == total:
                print(f" Ekstraksi: {i}/{total} file ({i/total*100:.1f}%)")

    extracted_wavs = glob.glob(f"{BASE_DIR}/**/*.wav", recursive=True)
    print(f"Part {part_num} selesai! Total .wav sejauh ini: {len(extracted_wavs)}")

    if len(extracted_wavs) == 0:
        print(f"ERROR: Tidak ada file .wav ditemukan setelah ekstraksi part {part_num}!")
    else:
        with open(EXTRACT_FLAG, 'w') as f:
            f.write(f"done={len(extracted_wavs)}")
        print(f"Flag part {part_num} disimpan.")

    os.remove(zip_path)
    print(f"  Zip part {part_num} dihapus untuk hemat storage.")

# Mapping semua wav yang ada
all_wavs = glob.glob(f"{BASE_DIR}/**/*.wav", recursive=True)
if not all_wavs:
    print(" ERROR: File .wav tidak ditemukan!")
else:
    path_map = {os.path.basename(p): p for p in all_wavs}
    df['audio_path'] = df['path'].apply(lambda x: path_map.get(os.path.basename(x)))
    df = df.dropna(subset=['audio_path'])

    print(f"\n JUMLAH DATA VALID: {len(df)}")
    if len(df) > 0:
        df = df.head(45000)
        ds = Dataset.from_dict({
            "audio": df["audio_path"].tolist(),
            "sentence": df["text"].tolist()
        })
        ds = ds.cast_column("audio", Audio(sampling_rate=16000))
        print(" Dataset BERHASIL dibuat di Kaggle!")

# Model Setup

In [ ]:
MODEL_NAME = "facebook/mms-1b-all"

processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)

# 1. Konfigurasi 4-Bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

# 2. Muat Model 
model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    ignore_mismatched_sizes=True,
    quantization_config=bnb_config,
    device_map="auto"
)

# 3. Persiapkan LoRA
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=False  
)

peft_config = LoraConfig(
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj"],  
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none"
)

model = get_peft_model(model, peft_config)

# Next epoch
model = PeftModel.from_pretrained(
    model,
    f"gracecalista/mms-khotbah-lora-final",   # dari Hub
    is_trainable=True                       
)

model.print_trainable_parameters()

# 4. Config Stabilitas
model.enable_input_require_grads()  
model.config.use_cache = False      

# Preprocessing

In [ ]:
import re

def normalize_text(text):
    text = text.lower()
    text = text.replace("1", "satu")
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

ds = ds.map(lambda x: {"sentence": normalize_text(x["sentence"])})

In [ ]:
import numpy as np
import gc
import os
from datasets import concatenate_datasets, load_from_disk

def prepare_dataset(batch):
    audio = batch["audio"]
    array = np.array(audio["array"], dtype=np.float32)
    array = np.nan_to_num(array)
    sr = audio["sampling_rate"]

    batch["input_values"] = processor(
        array,
        sampling_rate=sr,
        return_attention_mask=False
    ).input_values[0].tolist()
    batch["labels"] = processor.tokenizer(batch["sentence"]).input_ids
    return batch

# ── Pakai /kaggle/temp, tidak kena limit output ──
CHUNK_SIZE = 5000
SAVE_DIR = "/kaggle/temp/ds_chunks"  # ← pindah ke sini
os.makedirs(SAVE_DIR, exist_ok=True)

for start in range(0, len(ds), CHUNK_SIZE):
    end = min(start + CHUNK_SIZE, len(ds))
    chunk_path = f"{SAVE_DIR}/chunk_{start}_{end}"
    
    if os.path.exists(chunk_path):
        print(f"Skip {start} - {end} (sudah ada)")
        continue
    
    print(f"Proses sample {start} - {end}...")
    chunk = ds.select(range(start, end)).map(
        prepare_dataset,
        remove_columns=ds.column_names,
        num_proc=1,
        batched=False,
        writer_batch_size=100,
        load_from_cache_file=False
    )
    chunk.save_to_disk(chunk_path)
    del chunk
    gc.collect()
    print(f" Tersimpan")

# ── Gabungkan ──
print("Menggabungkan semua chunk...")
chunk_paths = sorted([f"{SAVE_DIR}/{d}" for d in os.listdir(SAVE_DIR)])
chunks = [load_from_disk(p) for p in chunk_paths]
ds = concatenate_datasets(chunks)
del chunks
gc.collect()
print(f"Selesai! Total: {len(ds)} sample")

# ── Filter + split ──
ds = ds.filter(lambda x: len(x["input_values"]) > len(x["labels"]) * 2)
ds = ds.shuffle(seed=42).train_test_split(test_size=0.1)

In [ ]:
def is_valid(example):
    input_len = len(example["input_values"])
    label_len = len(example["labels"])
    return input_len > label_len * 2 

ds = ds.filter(is_valid)

# Trainer

In [ ]:
!pip install jiwer

import evaluate
import numpy as np

wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = np.argmax(pred.predictions, axis=-1)

    label_ids = pred.label_ids.copy()
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(label_ids, group_tokens=False)

    print("\nDEBUG")
    print("PRED:", pred_str[:2])
    print("LABEL:", label_str[:2])

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

In [ ]:
from transformers import EarlyStoppingCallback

class CustomDataCollatorCTCWithPadding:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, features):
        input_features = [{"input_values": f["input_values"]} for f in features]
        batch = self.processor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.pad(labels=label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch

class MemoryCleanupCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        gc.collect()
        torch.cuda.empty_cache()

training_args = TrainingArguments(
    output_dir="./mms-khotbah-lora",

    per_device_train_batch_size=16,
    gradient_accumulation_steps=8,
    num_train_epochs=3,

    learning_rate=2e-5,
    warmup_steps=100,

    fp16=True,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    eval_strategy="steps",
    eval_steps=50,
    save_steps=50,
    logging_steps=10,

    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,

    remove_unused_columns=False,
    report_to="none",

    push_to_hub=True,
    hub_model_id="gracecalista/mms-khotbah-lora-final",
    hub_strategy="all_checkpoints",

    max_grad_norm=1.0
)

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=ds["train"],
    eval_dataset=ds["test"],

    data_collator=CustomDataCollatorCTCWithPadding(processor),

    processing_class=processor.feature_extractor,

    compute_metrics=compute_metrics,

    callbacks=[MemoryCleanupCallback(), EarlyStoppingCallback(early_stopping_patience=3)],
)



print("🚀 Training dimulai...")
# First epoch
trainer.train() 

# Next epoch
CHECKPOINT_NAME = "checkpoint-900"  # --> menyesuaikan checkpoint terakhir

# Download checkpoint locally first (Trainer needs local path)
from huggingface_hub import snapshot_download

print(f" Downloading checkpoint {CHECKPOINT_NAME} from Hub...")
local_ckpt_path = snapshot_download(
    repo_id="gracecalista/mms-khotbah-lora-final",
    allow_patterns=f"{CHECKPOINT_NAME}/*",
    local_dir="./mms-khotbah-lora"
)

ckpt_path = f"./mms-khotbah-lora/{CHECKPOINT_NAME}"
trainer.train(resume_from_checkpoint=ckpt_path)


trainer.push_to_hub()